# Exploração: `fator_contribuinte.csv`

Objetivo: entender se os fatores contribuintes de uma ocorrência aeronáutica trazem sinal preditivo suficiente para melhorar a predição de `aeronave_nivel_dano`.

**Questão central:** conseguimos criar features a partir deste arquivo que ensinam o modelo algo genuinamente novo — especialmente sobre a classe DESTRUÍDA?

---

**Estrutura do arquivo:**
- `codigo_ocorrencia3` — chave de junção com `ocorrencia.csv`
- `fator_nome` — nome do fator (ex: JULGAMENTO DE PILOTAGEM, MANUTENÇÃO DA AERONAVE)
- `fator_aspecto` — aspecto do fator (ex: DESEMPENHO DO SER HUMANO, ASPECTO PSICOLÓGICO)
- `fator_condicionante` — condicionante (ex: OPERAÇÃO DA AERONAVE, INDIVIDUAL)
- `fator_area` — área de classificação (FATOR HUMANO, FATOR OPERACIONAL, FATOR MATERIAL, OUTRO)

In [1]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
df_fc = pd.read_csv('data/raw/CENIPA_FAB/fator_contribuinte.csv', sep=';', encoding='latin-1')
df_oc = pd.read_csv('data/raw/CENIPA_FAB/ocorrencia.csv', sep=';', encoding='latin-1')
df_ae = pd.read_csv('data/raw/CENIPA_FAB/aeronave.csv', sep=';', encoding='latin-1')

# Replicate the same join used in the main notebook
df = df_oc.merge(df_ae, left_on='codigo_ocorrencia', right_on='codigo_ocorrencia2', how='inner')
df['aeronave_nivel_dano'] = df['aeronave_nivel_dano'].replace('***', pd.NA)
df = df.dropna(subset=['aeronave_nivel_dano'])

df_fc['fator_condicionante'] = df_fc['fator_condicionante'].replace('***', pd.NA)

print(f'Ocorrências com severidade: {len(df):,}')
print(f'Registros de fatores contribuintes: {len(df_fc):,}')
print(f'Ocorrências únicas com fatores: {df_fc["codigo_ocorrencia3"].nunique():,}')

Ocorrências com severidade: 12,928
Registros de fatores contribuintes: 8,613
Ocorrências únicas com fatores: 2,122


---
## 1 — Cobertura: quais ocorrências têm fatores contribuintes?

In [3]:
df['tem_fator'] = df['codigo_ocorrencia'].isin(df_fc['codigo_ocorrencia3'])

coverage = df['tem_fator'].value_counts()
print(f'Com fator contribuinte:    {coverage[True]:,} ({coverage[True]/len(df):.1%})')
print(f'Sem fator contribuinte:    {coverage[False]:,} ({coverage[False]/len(df):.1%})')

fig = px.pie(
    names=['Sem fator contribuinte', 'Com fator contribuinte'],
    values=[coverage[False], coverage[True]],
    title='Cobertura de Fatores Contribuintes no Dataset',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

Com fator contribuinte:    2,135 (16.5%)
Sem fator contribuinte:    10,793 (83.5%)


In [4]:
sev_order = ['NENHUM', 'LEVE', 'SUBSTANCIAL', 'DESTRUÍDA']

cov_sev = (
    df.groupby('aeronave_nivel_dano')['tem_fator']
    .value_counts(normalize=True)
    .rename('proporção')
    .reset_index()
)
cov_sev = cov_sev[cov_sev['tem_fator'] == True].copy()
cov_sev['aeronave_nivel_dano'] = pd.Categorical(cov_sev['aeronave_nivel_dano'], categories=sev_order, ordered=True)
cov_sev = cov_sev.sort_values('aeronave_nivel_dano')

print('% de ocorrências com fator contribuinte, por severidade:')
for _, row in cov_sev.iterrows():
    print(f'  {row["aeronave_nivel_dano"]:<12} {row["proporção"]:.1%}')

fig = px.bar(
    cov_sev,
    x='aeronave_nivel_dano', y='proporção',
    title='% de Ocorrências COM Fator Contribuinte por Severidade',
    labels={'aeronave_nivel_dano': 'Severidade', 'proporção': 'Proporção com fator'},
    color='aeronave_nivel_dano',
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto='.1%'
)
fig.update_layout(yaxis_tickformat='.0%', showlegend=False)
fig.show()

% de ocorrências com fator contribuinte, por severidade:
  NENHUM       2.5%
  LEVE         7.9%
  SUBSTANCIAL  57.0%
  DESTRUÍDA    60.6%


In [5]:
dist = (
    df.groupby(['tem_fator', 'aeronave_nivel_dano'])
    .size()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x / x.sum())
    .rename('proporção')
    .reset_index()
)
dist['grupo'] = dist['tem_fator'].map({True: 'Com fator contribuinte', False: 'Sem fator contribuinte'})
dist['aeronave_nivel_dano'] = pd.Categorical(dist['aeronave_nivel_dano'], categories=sev_order, ordered=True)

fig = px.bar(
    dist.sort_values('aeronave_nivel_dano'),
    x='aeronave_nivel_dano', y='proporção',
    color='grupo', barmode='group',
    title='Distribuição de Severidade: Com vs Sem Fator Contribuinte',
    labels={'aeronave_nivel_dano': 'Severidade', 'proporção': 'Proporção', 'grupo': ''},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto='.1%'
)
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

---
## 2 — Análise dos Fatores: o que está sendo registrado?

Entendendo a distribuição por `fator_area`, `fator_aspecto` e `fator_nome`.

In [6]:
area_counts = df_fc['fator_area'].value_counts().reset_index()
area_counts.columns = ['fator_area', 'contagem']

fig = px.bar(
    area_counts,
    x='contagem', y='fator_area',
    orientation='h',
    title='Distribuição por Área do Fator Contribuinte',
    labels={'contagem': 'Nº de registros', 'fator_area': ''},
    color='contagem',
    color_continuous_scale='Blues',
    text_auto=True
)
fig.update_layout(showlegend=False)
fig.show()

In [7]:
nome_counts = df_fc['fator_nome'].value_counts().head(20).reset_index()
nome_counts.columns = ['fator_nome', 'contagem']

fig = px.bar(
    nome_counts.sort_values('contagem', ascending=True),
    x='contagem', y='fator_nome',
    orientation='h',
    title='Top 20 Fatores Contribuintes por Nome',
    labels={'contagem': 'Nº de registros', 'fator_nome': ''},
    color='contagem',
    color_continuous_scale='Blues',
    text_auto=True
)
fig.show()

In [8]:
n_fatores = df_fc.groupby('codigo_ocorrencia3').size().reset_index(name='n_fatores')
print('Fatores por ocorrência:')
print(n_fatores['n_fatores'].describe().round(2))

fig = px.histogram(
    n_fatores, x='n_fatores',
    title='Distribuição: Nº de Fatores Contribuintes por Ocorrência',
    labels={'n_fatores': 'Nº de fatores', 'count': 'Ocorrências'},
    color_discrete_sequence=['steelblue']
)
fig.show()

Fatores por ocorrência:
count   2122.000
mean       4.060
std        2.960
min        1.000
25%        2.000
50%        3.000
75%        5.000
max       22.000
Name: n_fatores, dtype: float64


---
## 3 — Fatores por Severidade: o que diferencia DESTRUÍDA?

Cruzando os fatores contribuintes com `aeronave_nivel_dano` para identificar quais fatores são mais associados à severidade máxima.

In [9]:
# Join contributing factors with severity
sev_map = df[['codigo_ocorrencia', 'aeronave_nivel_dano']].drop_duplicates()
df_fc_sev = df_fc.merge(
    sev_map, left_on='codigo_ocorrencia3', right_on='codigo_ocorrencia', how='inner'
)
print(f'Registros com severidade linkada: {len(df_fc_sev):,}')
print(df_fc_sev['aeronave_nivel_dano'].value_counts())

Registros com severidade linkada: 8,579
aeronave_nivel_dano
SUBSTANCIAL    5103
DESTRUÍDA      1939
LEVE            945
NENHUM          592
Name: count, dtype: int64


In [10]:
area_sev = (
    df_fc_sev.groupby(['aeronave_nivel_dano', 'fator_area'])
    .size()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x / x.sum())
    .rename('proporção')
    .reset_index()
)
area_sev['aeronave_nivel_dano'] = pd.Categorical(
    area_sev['aeronave_nivel_dano'], categories=sev_order, ordered=True
)

fig = px.bar(
    area_sev.sort_values('aeronave_nivel_dano'),
    x='aeronave_nivel_dano', y='proporção',
    color='fator_area', barmode='stack',
    title='Composição de Área do Fator por Severidade',
    labels={'aeronave_nivel_dano': 'Severidade', 'proporção': 'Proporção', 'fator_area': 'Área'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto='.1%'
)
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

In [11]:
# Which factors are disproportionately present in DESTRUÍDA vs NENHUM?
destruida = df_fc_sev[df_fc_sev['aeronave_nivel_dano'] == 'DESTRUÍDA']['fator_nome'].value_counts(normalize=True)
nenhum    = df_fc_sev[df_fc_sev['aeronave_nivel_dano'] == 'NENHUM']['fator_nome'].value_counts(normalize=True)

compare = pd.DataFrame({'DESTRUÍDA': destruida, 'NENHUM': nenhum}).fillna(0)
compare['lift'] = (compare['DESTRUÍDA'] / (compare['NENHUM'] + 1e-6)).round(2)
compare = compare.sort_values('lift', ascending=False)

print('Fatores mais associados a DESTRUÍDA vs NENHUM (lift):')
print(compare.head(15).to_string())

fig = px.bar(
    compare.head(15).reset_index().rename(columns={'fator_nome': 'Fator'}),
    x='lift', y='Fator',
    orientation='h',
    title='Fatores mais associados a DESTRUÍDA vs NENHUM (lift ratio)',
    labels={'lift': 'Lift (DESTRUÍDA / NENHUM)', 'Fator': ''},
    color='lift',
    color_continuous_scale='Reds',
    text_auto='.1f'
)
fig.show()

Fatores mais associados a DESTRUÍDA vs NENHUM (lift):
                                           DESTRUÍDA  NENHUM     lift
fator_nome                                                           
ILUSÕES VISUAIS                                0.005   0.000 5157.300
ÁLCOOL                                         0.004   0.000 3610.110
INFLUÊNCIAS EXTERNAS                           0.004   0.000 3610.110
SOBRECARGA DE TAREFAS                          0.002   0.000 2062.920
EQUIPAMENTO - CARACTERÍSTICAS ERGONÔMICAS      0.001   0.000 1031.460
ENFERMIDADE                                    0.001   0.000 1031.460
PRESENÇA DE AVE                                0.001   0.000  515.730
OBESIDADE                                      0.001   0.000  515.730
DOR                                            0.001   0.000  515.730
USO ILÍCITO DE DROGAS                          0.001   0.000  515.730
VESTIMENTA INADEQUADA                          0.001   0.000  515.730
DESORIENTAÇÃO                       

In [12]:
# Heatmap: top 15 factor names × severity
top_names = df_fc['fator_nome'].value_counts().head(15).index.tolist()

heat = (
    df_fc_sev[df_fc_sev['fator_nome'].isin(top_names)]
    .groupby(['fator_nome', 'aeronave_nivel_dano'])
    .size()
    .unstack(fill_value=0)
)
heat = heat.div(heat.sum(axis=1), axis=0)  # normalize per factor
heat = heat[sev_order]

fig = px.imshow(
    heat,
    labels=dict(x='Severidade', y='Fator Nome', color='Proporção'),
    title='Distribuição de Severidade por Fator (top 15)',
    color_continuous_scale='Blues',
    text_auto='.2f',
    aspect='auto'
)
fig.show()

---
## 4 — Feature Engineering: o que podemos criar?

Como este é um dataset one-to-many (uma ocorrência → múltiplos fatores), precisamos agregar antes de juntar ao dataset principal.

Candidatas a feature:
1. `tem_fator` — flag binária (0/1): a ocorrência tem algum fator registrado?
2. `n_fatores` — contagem total de fatores
3. `n_fator_humano` / `n_fator_operacional` / `n_fator_material` — contagem por área
4. `tem_fator_material` — flag se há algum fator de tipo MATERIAL (mecânico)
5. Top fatores individuais como flags binárias (presença/ausência)

In [13]:
# Aggregate contributing factors per occurrence
agg = df_fc.groupby('codigo_ocorrencia3').agg(
    n_fatores=('fator_nome', 'count'),
    n_fator_humano=('fator_area', lambda x: (x == 'FATOR HUMANO').sum()),
    n_fator_operacional=('fator_area', lambda x: (x == 'FATOR OPERACIONAL').sum()),
    n_fator_material=('fator_area', lambda x: (x == 'FATOR MATERIAL').sum()),
).reset_index()
agg['tem_fator_material'] = (agg['n_fator_material'] > 0).astype(int)

# Binary flags for top individual factors
top_factors = df_fc['fator_nome'].value_counts().head(10).index.tolist()
for f in top_factors:
    flag_col = 'fator_' + f.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c').replace('ê', 'e')[:30]
    present = df_fc[df_fc['fator_nome'] == f]['codigo_ocorrencia3'].unique()
    agg[flag_col] = agg['codigo_ocorrencia3'].isin(present).astype(int)

print('Aggregated features shape:', agg.shape)
print(agg.head())

Aggregated features shape: (2122, 16)
   codigo_ocorrencia3  n_fatores  n_fator_humano  n_fator_operacional  \
0               28256          4               0                    4   
1               28335          4               0                    4   
2               28355          3               0                    3   
3               28375          3               0                    2   
4               28377          3               0                    2   

   n_fator_material  tem_fator_material  fator_julgamento_de_pilotagem  \
0                 0                   0                              1   
1                 0                   0                              1   
2                 0                   0                              0   
3                 0                   0                              1   
4                 0                   0                              0   

   fator_aplicacao_de_comandos  fator_supervisao_gerencial  \
0               

In [14]:
# Join with main dataset
df_enriched = df.merge(
    agg, left_on='codigo_ocorrencia', right_on='codigo_ocorrencia3', how='left'
)

# Fill NaN for occurrences without contributing factors
fill_cols = [c for c in agg.columns if c != 'codigo_ocorrencia3']
df_enriched[fill_cols] = df_enriched[fill_cols].fillna(0)
df_enriched['tem_fator'] = (df_enriched['n_fatores'] > 0).astype(int)

print(f'Enriched dataset shape: {df_enriched.shape}')
print(f'tem_fator=1: {df_enriched["tem_fator"].sum()} ({df_enriched["tem_fator"].mean():.1%})')

Enriched dataset shape: (12928, 62)
tem_fator=1: 2135 (16.5%)


---
## 5 — Poder Preditivo das Features Candidatas

Antes de treinar qualquer modelo, vamos verificar se as novas features de fato separam as classes de severidade.

In [15]:
# tem_fator × severity
tf_sev = (
    df_enriched.groupby(['tem_fator', 'aeronave_nivel_dano'])
    .size()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x / x.sum())
    .rename('proporção')
    .reset_index()
)
tf_sev['grupo'] = tf_sev['tem_fator'].map({0: 'Sem fator', 1: 'Com fator'})
tf_sev['aeronave_nivel_dano'] = pd.Categorical(tf_sev['aeronave_nivel_dano'], categories=sev_order, ordered=True)

fig = px.bar(
    tf_sev.sort_values('aeronave_nivel_dano'),
    x='aeronave_nivel_dano', y='proporção',
    color='grupo', barmode='group',
    title='Poder preditivo de `tem_fator`: distribuição de severidade por grupo',
    labels={'aeronave_nivel_dano': 'Severidade', 'proporção': 'Proporção', 'grupo': ''},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto='.1%'
)
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

In [16]:
plot_df = df_enriched[df_enriched['tem_fator'] == 1].copy()
plot_df['aeronave_nivel_dano'] = pd.Categorical(
    plot_df['aeronave_nivel_dano'], categories=sev_order, ordered=True
)

fig = px.box(
    plot_df.sort_values('aeronave_nivel_dano'),
    x='aeronave_nivel_dano', y='n_fatores',
    color='aeronave_nivel_dano',
    title='Nº de Fatores Contribuintes por Severidade (ocorrências com fatores)',
    labels={'aeronave_nivel_dano': 'Severidade', 'n_fatores': 'Nº de fatores'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

In [17]:
# Does tem_fator_material separate severity?
mat_sev = (
    df_enriched[df_enriched['tem_fator'] == 1]
    .groupby(['tem_fator_material', 'aeronave_nivel_dano'])
    .size()
    .groupby(level=0, group_keys=False)
    .apply(lambda x: x / x.sum())
    .rename('proporção')
    .reset_index()
)
mat_sev['grupo'] = mat_sev['tem_fator_material'].map({0: 'Sem fator material', 1: 'Com fator material'})
mat_sev['aeronave_nivel_dano'] = pd.Categorical(mat_sev['aeronave_nivel_dano'], categories=sev_order, ordered=True)

fig = px.bar(
    mat_sev.sort_values('aeronave_nivel_dano'),
    x='aeronave_nivel_dano', y='proporção',
    color='grupo', barmode='group',
    title='Poder preditivo de `tem_fator_material` (entre ocorrências com fatores)',
    labels={'aeronave_nivel_dano': 'Severidade', 'proporção': 'Proporção', 'grupo': ''},
    color_discrete_sequence=px.colors.qualitative.Set2,
    text_auto='.1%'
)
fig.update_layout(yaxis_tickformat='.0%')
fig.show()

---
## 6 — Avaliação de Viabilidade

Antes de levar qualquer feature para o modelo principal, precisamos responder:

1. **O sinal existe?** Os fatores separam DESTRUÍDA das outras classes?
2. **A cobertura é suficiente?** 83.5% das ocorrências têm `tem_fator=0` — isso é sinal negativo, não ausência de sinal.
3. **O preenchimento por 0 faz sentido?** Ausência de registro de fator ≠ ausência de fator real.
4. **Há vazamento de dados?** Fatores contribuintes são determinados *após* a investigação do acidente — o modelo pode estar aprendendo do futuro.

In [18]:
# Data leakage risk analysis
# If contributing factors are only registered for investigated accidents,
# and if investigation depth correlates with severity, we have a confound.

print('=== Distribuição de severidade por tem_fator ===')
print('\nCOM fator contribuinte:')
print(df_enriched[df_enriched['tem_fator']==1]['aeronave_nivel_dano'].value_counts(normalize=True).round(3))
print('\nSEM fator contribuinte:')
print(df_enriched[df_enriched['tem_fator']==0]['aeronave_nivel_dano'].value_counts(normalize=True).round(3))

print('\n=== Interpretação ===')
print('SUBSTANCIAL + DESTRUÍDA com fator:', round(
    df_enriched[df_enriched['tem_fator']==1]['aeronave_nivel_dano'].isin(['SUBSTANCIAL','DESTRUÍDA']).mean(), 3
))
print('SUBSTANCIAL + DESTRUÍDA sem fator:', round(
    df_enriched[df_enriched['tem_fator']==0]['aeronave_nivel_dano'].isin(['SUBSTANCIAL','DESTRUÍDA']).mean(), 3
))

=== Distribuição de severidade por tem_fator ===

COM fator contribuinte:
aeronave_nivel_dano
SUBSTANCIAL   0.645
DESTRUÍDA     0.156
LEVE          0.123
NENHUM        0.076
Name: proportion, dtype: float64

SEM fator contribuinte:
aeronave_nivel_dano
NENHUM        0.599
LEVE          0.285
SUBSTANCIAL   0.096
DESTRUÍDA     0.020
Name: proportion, dtype: float64

=== Interpretação ===
SUBSTANCIAL + DESTRUÍDA com fator: 0.8
SUBSTANCIAL + DESTRUÍDA sem fator: 0.116


---
## 7 — Conclusão e Próximo Passo

Com base nesta exploração, responda:

**As features de fator contribuinte são seguras para usar no modelo principal?**

Considere:
- Se o sinal existe e a cobertura é razoável → integrar ao `mvp_aviation_severity.ipynb` como novas features numéricas
- Se há risco de vazamento de dados → usar apenas `tem_fator` (flag binária inofensiva) e descartar as específicas
- Se o sinal é fraco → documentar o resultado negativo no rationale e seguir sem essas features

**Features candidatas para integração (se aprovadas):**
```python
FEAT_NUM_P2 = ['tem_fator', 'n_fatores', 'n_fator_humano', 'n_fator_operacional', 'n_fator_material']
```
Essas são numéricas e se encaixam diretamente no pipeline como um segundo transformer (sem OHE necessário).